In [10]:
import os
data_path = '../../data/'

# List all CSV files
csv_files = [f for f in os.listdir(data_path) if f.endswith('.csv')]
print("CSV files found:", csv_files)

# Try loading with encoding
print("Loading UNSW-NB15 training data...", end=" ")
df_train = pd.read_csv(os.path.join(data_path, 'UNSW_NB15_training-set.csv'), encoding='latin-1')
print(f"✅ Shape: {df_train.shape}")

print("Loading UNSW-NB15 testing data...", end=" ")
df_test = pd.read_csv(os.path.join(data_path, 'UNSW_NB15_testing-set.csv'), encoding='latin-1')
print(f"✅ Shape: {df_test.shape}")

# Combine
df = pd.concat([df_train, df_test], ignore_index=True)
print(f"\n📊 Combined dataset shape: {df.shape}")
print(f"Total rows: {len(df):,}")

CSV files found: ['NUSW-NB15_features.csv', 'UNSW_NB15_testing-set.csv', 'UNSW_NB15_training-set.csv']
Loading UNSW-NB15 training data... ✅ Shape: (82332, 45)
Loading UNSW-NB15 testing data... ✅ Shape: (175341, 45)

📊 Combined dataset shape: (257673, 45)
Total rows: 257,673


In [11]:
# ==============================================
# 2. DATA EXPLORATION & QUALITY CHECKS
# ==============================================

print("Data Info:")
print(df.info())

print("\n" + "="*80)
print("Missing Values:")
print(df.isnull().sum())

print("\n" + "="*80)
print("Statistical Summary:")
print(df.describe())

print("\n" + "="*80)
print("Target Variable (label) Distribution:")
print(df['label'].value_counts())

Data Info:
<class 'pandas.DataFrame'>
RangeIndex: 257673 entries, 0 to 257672
Data columns (total 45 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   ï»¿id              257673 non-null  int64  
 1   dur                257673 non-null  float64
 2   proto              257673 non-null  str    
 3   service            257673 non-null  str    
 4   state              257673 non-null  str    
 5   spkts              257673 non-null  int64  
 6   dpkts              257673 non-null  int64  
 7   sbytes             257673 non-null  int64  
 8   dbytes             257673 non-null  int64  
 9   rate               257673 non-null  float64
 10  sttl               257673 non-null  int64  
 11  dttl               257673 non-null  int64  
 12  sload              257673 non-null  float64
 13  dload              257673 non-null  float64
 14  sloss              257673 non-null  int64  
 15  dloss              257673 non-null  int64  
 16  si

In [12]:
# ==============================================
# Check for Outliers, Duplicates, and Data Quality
# ==============================================

print("1. DUPLICATES CHECK:")
print(f"Total duplicate rows: {df.duplicated().sum()}")
print(f"Percentage: {df.duplicated().sum() / len(df) * 100:.2f}%")

print("\n2. INFINITE VALUES CHECK:")
print(f"Total infinite values: {np.isinf(df.select_dtypes(include=[np.number])).sum().sum()}")

print("\n3. DATA TYPES:")
print(df.dtypes)

print("\n4. TARGET VARIABLE DISTRIBUTION:")
print(df['label'].value_counts())
print("\nClass Distribution (%):")
print((df['label'].value_counts() / len(df) * 100).round(2))

print("\n5. OUTLIERS CHECK (using IQR method on first numeric column):")
numeric_cols = df.select_dtypes(include=[np.number]).columns
if len(numeric_cols) > 0:
    col = numeric_cols[0]
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)]
    print(f"Column: {col}")
    print(f"Outliers found: {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)")

1. DUPLICATES CHECK:
Total duplicate rows: 0
Percentage: 0.00%

2. INFINITE VALUES CHECK:
Total infinite values: 0

3. DATA TYPES:
ï»¿id                  int64
dur                  float64
proto                    str
service                  str
state                    str
spkts                  int64
dpkts                  int64
sbytes                 int64
dbytes                 int64
rate                 float64
sttl                   int64
dttl                   int64
sload                float64
dload                float64
sloss                  int64
dloss                  int64
sinpkt               float64
dinpkt               float64
sjit                 float64
djit                 float64
swin                   int64
stcpb                  int64
dtcpb                  int64
dwin                   int64
tcprtt               float64
synack               float64
ackdat               float64
smean                  int64
dmean                  int64
trans_depth            int64

In [13]:
# Fix BOM issue and reload
print("Loading with BOM fix...", end=" ")
df_train = pd.read_csv('../../data/UNSW_NB15_training-set.csv', encoding='utf-8-sig')
df_test = pd.read_csv('../../data/UNSW_NB15_testing-set.csv', encoding='utf-8-sig')
df = pd.concat([df_train, df_test], ignore_index=True)
print(f"✅ Shape: {df.shape}")

print("\nColumn names after BOM fix:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

Loading with BOM fix... ✅ Shape: (257673, 45)

Column names after BOM fix:
['id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'attack_cat', 'label']

Data types:
id                     int64
dur                  float64
proto                    str
service                  str
state                    str
spkts                  int64
dpkts                  int64
sbytes                 int64
dbytes                 int64
rate                 float64
sttl                   int64
dttl                   int64
sload                float64
dload             